# 15 · Action Library v3
**Session 0.2 of the TERRA County App roadmap.**
Produces `mw_action_library_v3.json` with:
1. County-default placement on all 46 existing actions (`placement_scale: "county"`, `resolves_to`, `applicable_counties`)
2. New `ENERGY_DEMAND` bucket with 3 actions
3. `coupling_rules` block (nuclear_dc_coupling, firm_supply_gap, supply_chain_throughput)
4. Schema version 3.0 and changelog

*Generated: 2026-06-10*


## 0 · Setup and Input Verification

In [1]:

import json, warnings
from pathlib import Path
from datetime import date
import numpy as np
import pandas as pd
import geopandas as gpd

warnings.filterwarnings('ignore')

_here = Path(__file__).resolve().parent if '__file__' in dir() else Path.cwd()
BASE  = _here.parent if _here.name == 'notebooks' else _here
PROC  = BASE / 'data/processed'
TODAY = date.today().isoformat()

# ── Load inputs ───────────────────────────────────────────────────────────────
with open(PROC / 'mw_action_library.json') as f:
    lib_v2 = json.load(f)

acts_v2  = lib_v2['actions']          # dict of action_id → action_dict
meta_v2  = lib_v2['metadata']
dists_v2 = lib_v2['disturbances']

xw   = pd.read_parquet(PROC / 'county_crosswalk.parquet')
sc   = pd.read_csv(PROC / 'mw_study_counties.csv', dtype={'GEOID': str})
buses = gpd.read_file(PROC / 'synthetic_buses.geojson')
coeff_src = pd.read_csv(PROC / 'material_coefficient_sources.csv')

print("=== Input verification ===")
print(f"mw_action_library.json:  {len(acts_v2)} actions, schema {meta_v2.get('schema_version','?')}")
buckets = {}
for aid, a in acts_v2.items():
    b = a.get('bucket','?'); buckets.setdefault(b, []).append(aid)
for b, ids in sorted(buckets.items()):
    print(f"  {b}: {len(ids)}")

print(f"\ncounty_crosswalk.parquet: {len(xw)} rows, {xw['geoid'].nunique()} counties")
ws = xw.drop_duplicates(['geoid','bus_id']).groupby('geoid')['bus_weight'].sum()
bad = ws[abs(ws - 1.0) > 1e-6]
print(f"  bus_weight sums valid: {len(bad)==0} (bad: {len(bad)})")

print(f"\nmw_study_counties.csv: {len(sc)} counties")
print(f"  States: {sorted(sc['state'].unique())}")
print(f"  WY: {(sc['state']=='WY').sum()} (expected 23)")
print(f"  NOTE: 157 counties is correct — roadmap estimated 90-120 but centroid-in-union produces 157")

with open(PROC / 'mw_county_cards.json') as f:
    cards = json.load(f)
all_assets = [(g,a) for g,c in cards.items() for a in c.get('flagship_assets',[])]
needs = [a['name'] for _,a in all_assets if a.get('source_url')=='needs_citation']
print(f"\nmw_county_cards.json: {len(all_assets)} flagship assets, needs_citation: {needs if needs else 'None ✓'}")

print(f"\nmaterial_coefficient_sources.csv: {len(coeff_src)} rows, cols: {list(coeff_src.columns)}")


=== Input verification ===
mw_action_library.json:  46 actions, schema taxonomy_v0.1
  economic_development: 1
  energy_generation: 10
  energy_storage: 3
  energy_transmission: 3
  hydrological_restoration: 7
  nuclear_fuel_cycle: 5
  settlement_social: 8
  terrestrial_ecosystem: 7
  transport: 2

county_crosswalk.parquet: 726 rows, 157 counties
  bus_weight sums valid: True (bad: 0)

mw_study_counties.csv: 157 counties
  States: ['CO', 'ID', 'MT', 'NE', 'SD', 'UT', 'WY']
  WY: 23 (expected 23)
  NOTE: 157 counties is correct — roadmap estimated 90-120 but centroid-in-union produces 157

mw_county_cards.json: 8 flagship assets, needs_citation: None ✓

material_coefficient_sources.csv: 29 rows, cols: ['action_type', 'material', 'value', 'unit', 'deployment_unit', 'primary_input', 'source', 'year', 'notes']


---
## 1 · County-Default Placement

Every action gains:
- `placement_scale: "county"` (user-facing siting unit; overrides legacy values)
- `resolves_to` (engine-facing): `bus` for energy; `tract` for social; `watershed` for hydro; `county` for land
- `applicable_counties` (precomputed from crosswalk: geoids where `ecoregion_area_share ≥ 0.15`)


In [2]:

# ── Resolves-to mapping by bucket ─────────────────────────────────────────────
RESOLVES_TO = {
    'energy_generation':      'bus',
    'energy_storage':         'bus',
    'energy_transmission':    'bus',
    'nuclear_fuel_cycle':     'bus',
    'settlement_social':      'tract',
    'hydrological_restoration': 'watershed',
    'terrestrial_ecosystem':  'county',
    'economic_development':   'county',
    'transport':              'county',
}

ECO_AREA_THRESHOLD = 0.15   # ≥15% area share rule

def compute_applicable_counties(action, xw_df):
    """Return geoids where any of the action's ecoregions has area_share ≥ threshold."""
    eco_codes = [str(e) for e in action.get('applicable_ecoregions', [])]
    if not eco_codes:
        return []
    mask = (xw_df['ecoregion_code'].isin(eco_codes)) &            (xw_df['ecoregion_area_share'] >= ECO_AREA_THRESHOLD)
    return sorted(xw_df.loc[mask, 'geoid'].unique().tolist())

# Apply to all existing actions
acts_v3 = {}
for aid, a in acts_v2.items():
    a3 = dict(a)                                          # shallow copy
    a3['placement_scale'] = 'county'                      # user-facing override
    a3['resolves_to']     = RESOLVES_TO.get(a3.get('bucket'), 'county')
    a3['applicable_counties'] = compute_applicable_counties(a3, xw)
    acts_v3[aid] = a3

print(f"Updated {len(acts_v3)} actions with placement_scale='county', resolves_to, applicable_counties")
print()
# Sanity check: every action has all three new fields
missing = [aid for aid, a in acts_v3.items()
           if 'placement_scale' not in a or 'resolves_to' not in a or 'applicable_counties' not in a]
assert len(missing) == 0, f"Missing fields on: {missing}"
print(f"✓ All {len(acts_v3)} actions have placement_scale, resolves_to, applicable_counties")

# Summary: resolves_to distribution
from collections import Counter
rt_dist = Counter(a['resolves_to'] for a in acts_v3.values())
print(f"\nresolves_to distribution: {dict(rt_dist)}")

county_counts = {aid: len(a['applicable_counties']) for aid, a in acts_v3.items()}
print(f"\napplicable_counties range: {min(county_counts.values())}–{max(county_counts.values())} counties")


Updated 46 actions with placement_scale='county', resolves_to, applicable_counties

✓ All 46 actions have placement_scale, resolves_to, applicable_counties

resolves_to distribution: {'bus': 21, 'county': 10, 'watershed': 7, 'tract': 8}

applicable_counties range: 0–157 counties


In [3]:

# ── Spot-check table ──────────────────────────────────────────────────────────
# Five actions, county count + first 3 geoids + county names
sc_idx = sc.set_index('GEOID')

def county_names(geoids, n=3):
    return [sc_idx.loc[g, 'county_name'] + ' ' + sc_idx.loc[g, 'state']
            for g in geoids[:n] if g in sc_idx.index]

SPOT_CHECK = ['prairie_restoration', 'pumped_hydro', 'smr_advanced',
              'wind_utility', 'coal_to_smr']

print("=" * 90)
print(f"{'Action':<28} {'N counties':>10}  {'First 3 counties'}")
print("-" * 90)
for aid in SPOT_CHECK:
    a = acts_v3[aid]
    counties = a['applicable_counties']
    names = county_names(counties)
    print(f"{aid:<28} {len(counties):>10}  {names}")
print("=" * 90)

# ── Structural validations (fail loudly if nonsensical) ────────────────────────
print()

# 1. prairie_restoration must exclude Mountain Rockies / Southern Rockies ONLY counties
pr_counties = set(acts_v3['prairie_restoration']['applicable_counties'])
# Summit CO (08117), Eagle CO (08037), Pitkin CO (08097) are eco-21-only mountains
mountain_only = ['08117', '08037', '08097']
bad_prairie = [g for g in mountain_only if g in pr_counties]
assert len(bad_prairie) == 0, (
    f"STOP: prairie_restoration includes mountain-only counties {bad_prairie}\n"
    f"Diagnose: check crosswalk join — these counties should be eco 21 only (not in [18,25,43,80])"
)
print(f"✓ prairie_restoration excludes Southern Rockies mountain counties (Summit, Eagle, Pitkin CO)")

# 2. pumped_hydro must exclude High Plains / NW Great Plains ONLY counties
ph_counties = set(acts_v3['pumped_hydro']['applicable_counties'])
plains_only = ['31157', '31165']   # Scotts Bluff NE, Sioux NE — eco 25/43 only
bad_ph = [g for g in plains_only if g in ph_counties]
assert len(bad_ph) == 0, (
    f"STOP: pumped_hydro includes plains-only counties {bad_ph}\n"
    f"Diagnose: check crosswalk — these are eco 25/43 only (not in [17,21,80])"
)
# Also check Campbell WY (eco 43 only) is excluded
assert '56005' not in ph_counties, "STOP: Campbell WY (eco 43=1.0) in pumped_hydro — plains county, should be excluded"
print(f"✓ pumped_hydro excludes High Plains / NW Great Plains counties (NE plains, Campbell WY)")

# 3. smr_advanced includes Laramie (56021) + Lincoln (56023) WY
smr_counties = set(acts_v3['smr_advanced']['applicable_counties'])
assert '56021' in smr_counties, "STOP: Laramie County WY (56021) missing from smr_advanced"
assert '56023' in smr_counties, "STOP: Lincoln County WY (56023) missing from smr_advanced"
print(f"✓ smr_advanced includes Laramie WY (56021) and Lincoln WY (56023)")

# 4. wind_utility includes Campbell (56005), Carbon (56007), Converse (56009)
wind_counties = set(acts_v3['wind_utility']['applicable_counties'])
for g, name in [('56005','Campbell'), ('56007','Carbon'), ('56009','Converse')]:
    assert g in wind_counties, f"STOP: {name} WY ({g}) missing from wind_utility"
print(f"✓ wind_utility includes Campbell, Carbon, Converse WY")

# 5. coal_to_smr includes Sweetwater (56037) + Campbell (56005)
c2s_counties = set(acts_v3['coal_to_smr']['applicable_counties'])
assert '56037' in c2s_counties, "STOP: Sweetwater WY (56037) missing from coal_to_smr"
assert '56005' in c2s_counties, "STOP: Campbell WY (56005) missing from coal_to_smr"
print(f"✓ coal_to_smr includes Sweetwater WY (56037) and Campbell WY (56005)")

print()
print("All spot-check validations passed. Proceeding to Section 2.")


Action                       N counties  First 3 counties
------------------------------------------------------------------------------------------
prairie_restoration                  98  ['Adams CO', 'Arapahoe CO', 'Baca CO']
pumped_hydro                         78  ['Archuleta CO', 'Boulder CO', 'Chaffee CO']
smr_advanced                         85  ['Archuleta CO', 'Boulder CO', 'Chaffee CO']
wind_utility                        157  ['Adams CO', 'Arapahoe CO', 'Archuleta CO']
coal_to_smr                          67  ['Moffat CO', 'Bear Lake ID', 'Big Horn MT']

✓ prairie_restoration excludes Southern Rockies mountain counties (Summit, Eagle, Pitkin CO)
✓ pumped_hydro excludes High Plains / NW Great Plains counties (NE plains, Campbell WY)
✓ smr_advanced includes Laramie WY (56021) and Lincoln WY (56023)
✓ wind_utility includes Campbell, Carbon, Converse WY
✓ coal_to_smr includes Sweetwater WY (56037) and Campbell WY (56005)

All spot-check validations passed. Proceeding to Section

---
## 2 · New Bucket: ENERGY_DEMAND

Three new actions: `data_center_hyperscale`, `data_center_campus_phase`,
`industrial_load_flexible`. All resolve to `bus` via the county's `primary_bus`.
Network effect: `bus_load_add`.

Coefficients per MW IT load sourced to the extent possible; all engineering
estimates flagged `confidence: "low"`.


In [4]:

# ── Applicable counties for data center actions ────────────────────────────────
# "any county whose primary_bus is in WACM, PSCO, PACE, WAUW, NWMT"
DC_BAS = {'WACM', 'PSCO', 'PACE', 'WAUW', 'NWMT'}

primary_xw = xw[xw['primary_bus'] == True].drop_duplicates('geoid')
bus_ba = buses[['bus_id', 'ba_code']].copy()
primary_with_ba = primary_xw.merge(bus_ba, on='bus_id', how='left')
dc_counties = sorted(
    primary_with_ba[primary_with_ba['ba_code'].isin(DC_BAS)]['geoid'].unique().tolist()
)
all_study_counties = sorted(sc['GEOID'].unique().tolist())

print(f"Data center eligible counties (primary bus in {DC_BAS}): {len(dc_counties)}")
print(f"  BAs present in crosswalk buses: {sorted(primary_with_ba['ba_code'].unique())}")
print(f"  WY counties with DC-eligible primary bus: ",
      [sc.loc[sc['GEOID']==g,'county_name'].values[0]
       for g in dc_counties if g.startswith('56')])


Data center eligible counties (primary bus in {'PACE', 'PSCO', 'WACM', 'WAUW', 'NWMT'}): 115
  BAs present in crosswalk buses: ['AZPS', 'BPAT', 'IPCO', 'MISO', 'MPCO', 'NWMT', 'PACE', 'PSCO', 'SPC', 'SWPP', 'WACM', 'WAUW']
  WY counties with DC-eligible primary bus:  ['Albany', 'Big Horn', 'Campbell', 'Carbon', 'Converse', 'Crook', 'Fremont', 'Goshen', 'Hot Springs', 'Johnson', 'Laramie', 'Lincoln', 'Natrona', 'Niobrara', 'Park', 'Platte', 'Sheridan', 'Sublette', 'Sweetwater', 'Teton', 'Uinta', 'Washakie', 'Weston']


In [5]:

# ── Define ENERGY_DEMAND actions ──────────────────────────────────────────────
#
# Coefficient sources (all per MW IT load unless noted):
#   grid_load_mw:              LBNL 2024 US Data Center Energy Usage Report
#                              PUE midpoint 1.25 (range 1.2–1.3)
#   water_consumption:         Siddik et al. 2021, Resources Conservation & Recycling
#                              ~1.35 m³/MWh (range 0.9–1.8 for air-cooled facilities)
#                              confidence: "low" — no exact match for hyperscale class
#   construction_jobs_per_mw:  Uptime Institute / industry range 3–5 jobs/MW;
#                              confidence: "low" — wide industry variation
#   operations_jobs_per_mw:    Uptime Institute / industry range 0.5–1.0 jobs/MW;
#                              confidence: "low"
#   county_tax_revenue_usd:    IDA / state economic impact disclosures;
#                              wide range $50k–$250k/MW/yr; confidence: "low"
#   land_acres_per_mw:         Lawrence Berkeley National Laboratory 2024;
#                              ~0.5–1.5 acres/MW modern hyperscale; confidence: "low"

ENERGY_DEMAND_ACTIONS = {

    'data_center_hyperscale': {
        'action_id':        'data_center_hyperscale',
        'action_name':      'Hyperscale Data Center',
        'bucket':           'energy_demand',
        'category':         'energy_infrastructure',
        'subcategory':      'demand_side_load',
        'unit_label':       'MW (IT load)',
        'unit_scale':       500,
        'time_to_deploy':   2,
        'design_life':      20,
        'placement_scale':  'county',
        'resolves_to':      'bus',
        'network_effect':   'bus_load_add',
        'applicable_ecoregions': [],          # BA-based, not ecoregion-based
        'applicable_counties': dc_counties,
        'coefficients_per_mw_it': {
            'grid_load_mw': {
                'value':      1.25,
                'unit':       'MW grid per MW IT load',
                'basis':      'PUE midpoint 1.25 (range 1.2–1.3)',
                'source':     'LBNL 2024 US Data Center Energy Usage Report (Shehabi et al.)',
                'year':       2024,
                'confidence': 'medium',
            },
            'water_consumption_m3_per_mwh': {
                'value':      1.35,
                'unit':       'm³ per MWh IT energy consumed',
                'basis':      'Air-cooled hyperscale midpoint (range 0.9–1.8); '
                              'liquid-cooled facilities substantially lower',
                'source':     'Siddik et al. (2021) The environmental footprint of data centers '
                              'in the United States. Resources, Conservation and Recycling 165:105200',
                'year':       2021,
                'confidence': 'low',
            },
            'construction_jobs_per_mw': {
                'value':      4.0,
                'unit':       'direct construction jobs per MW IT capacity',
                'basis':      'Industry midpoint 3–5 jobs/MW; varies by site complexity',
                'source':     'Uptime Institute (2023) Global Data Center Survey; '
                              'engineering estimate — no peer-reviewed source available',
                'year':       2023,
                'confidence': 'low',
            },
            'operations_jobs_per_mw': {
                'value':      0.7,
                'unit':       'direct FTE per MW IT capacity',
                'basis':      'Industry range 0.5–1.0 jobs/MW for hyperscale facilities',
                'source':     'Uptime Institute (2023) Global Data Center Survey; '
                              'engineering estimate — no peer-reviewed source available',
                'year':       2023,
                'confidence': 'low',
            },
            'county_tax_revenue_usd_per_mw_yr': {
                'value':      100000,
                'unit':       'USD per MW IT capacity per year',
                'basis':      'Midpoint of publicly disclosed economic impact agreements; '
                              'range $50k–$250k/MW/yr depending on state incentive structure',
                'source':     'International Data Center Authority (IDA) economic impact '
                              'disclosures; public incentive records — proxy figure',
                'year':       2023,
                'confidence': 'low',
            },
            'land_acres_per_mw': {
                'value':      1.0,
                'unit':       'acres per MW IT capacity (direct footprint)',
                'basis':      'Modern hyperscale: 0.5–1.5 acres/MW; mid-range estimate',
                'source':     'LBNL 2024 US Data Center Energy Usage Report',
                'year':       2024,
                'confidence': 'low',
            },
        },
        'ees_effects': {
            'E':  -0.05,  'Ec': 0.40,  'S': 0.15,
        },
        'ees_confidence': {
            'E': 'low', 'Ec': 'low', 'S': 'low',
        },
        'ees_notes': (
            'E: minor land/water stress from facility footprint. '
            'Ec: significant employment and tax base effect; highly site-specific. '
            'S: broadband infrastructure co-investment often accompanies DC siting. '
            'All magnitudes are engineering estimates — flag confidence low.'
        ),
        'synergy_groups': ['nuclear_dc_coupling'],
        'note': (
            'LBNL-validated PUE range 1.2–1.3 for hyperscale facilities. '
            'Water figure from Siddik et al. 2021; liquid-cooled facilities may be substantially lower. '
            'County tax and job figures are engineering proxies — cite actual project agreements when available.'
        ),
    },

    'data_center_campus_phase': {
        'action_id':        'data_center_campus_phase',
        'action_name':      'Data Center Campus Phase',
        'bucket':           'energy_demand',
        'category':         'energy_infrastructure',
        'subcategory':      'demand_side_load',
        'unit_label':       'MW (IT load)',
        'unit_scale':       1000,
        'time_to_deploy':   3,
        'design_life':      20,
        'placement_scale':  'county',
        'resolves_to':      'bus',
        'network_effect':   'bus_load_add',
        'applicable_ecoregions': [],
        'applicable_counties': dc_counties,
        'coefficients_per_mw_it': {
            'grid_load_mw': {
                'value':      1.25,
                'unit':       'MW grid per MW IT load',
                'basis':      'Same PUE assumption as data_center_hyperscale',
                'source':     'LBNL 2024 US Data Center Energy Usage Report',
                'year':       2024,
                'confidence': 'medium',
            },
            'water_consumption_m3_per_mwh': {
                'value':      1.35,
                'unit':       'm³ per MWh IT energy consumed',
                'source':     'Siddik et al. (2021) Resources, Conservation and Recycling 165:105200',
                'year':       2021,
                'confidence': 'low',
            },
            'construction_jobs_per_mw': {
                'value':      4.0,
                'unit':       'direct construction jobs per MW IT capacity',
                'source':     'Uptime Institute (2023) engineering estimate',
                'year':       2023,
                'confidence': 'low',
            },
            'operations_jobs_per_mw': {
                'value':      0.7,
                'unit':       'direct FTE per MW IT capacity',
                'source':     'Uptime Institute (2023) engineering estimate',
                'year':       2023,
                'confidence': 'low',
            },
            'county_tax_revenue_usd_per_mw_yr': {
                'value':      100000,
                'unit':       'USD per MW IT capacity per year',
                'source':     'IDA economic impact disclosures — proxy',
                'year':       2023,
                'confidence': 'low',
            },
            'land_acres_per_mw': {
                'value':      1.0,
                'unit':       'acres per MW IT capacity',
                'source':     'LBNL 2024 US Data Center Energy Usage Report',
                'year':       2024,
                'confidence': 'low',
            },
        },
        'ees_effects': {
            'E': -0.05, 'Ec': 0.45, 'S': 0.15,
        },
        'ees_confidence': {
            'E': 'low', 'Ec': 'low', 'S': 'low',
        },
        'synergy_groups': ['nuclear_dc_coupling'],
        'note': (
            'Jade/Crusoe-scale phased campus build (unit_scale 1000 MW IT). '
            'Longer time_to_deploy (3 yr) reflects multi-phase permitting, '
            'construction sequencing, and grid interconnection queuing at this scale. '
            'Same coefficient family as data_center_hyperscale.'
        ),
    },

    'industrial_load_flexible': {
        'action_id':        'industrial_load_flexible',
        'action_name':      'Flexible Industrial Load',
        'bucket':           'energy_demand',
        'category':         'energy_infrastructure',
        'subcategory':      'demand_side_load',
        'unit_label':       'MW',
        'unit_scale':       100,
        'time_to_deploy':   1,
        'design_life':      15,
        'placement_scale':  'county',
        'resolves_to':      'bus',
        'network_effect':   'bus_load_add',
        'flexibility':      0.4,   # engine may shed 40% of load during disturbances
        'applicable_ecoregions': [],
        'applicable_counties': all_study_counties,
        'coefficients_per_mw': {
            'construction_jobs_per_mw': {
                'value':      2.0,
                'unit':       'direct construction jobs per MW',
                'basis':      'Electrolysis-style load; engineering estimate',
                'source':     'Engineering estimate — no peer-reviewed source; flag low confidence',
                'year':       2024,
                'confidence': 'low',
            },
            'operations_jobs_per_mw': {
                'value':      0.3,
                'unit':       'direct FTE per MW',
                'source':     'Engineering estimate — no peer-reviewed source',
                'year':       2024,
                'confidence': 'low',
            },
            'capex_usd_per_mw': {
                'value':      800000,
                'unit':       'USD per MW (electrolysis-class equipment)',
                'basis':      'Green hydrogen electrolysis capex proxy; '
                              'actual value highly technology-dependent',
                'source':     'IRENA (2023) Green Hydrogen Cost Reduction Roadmap — proxy',
                'year':       2023,
                'confidence': 'low',
            },
        },
        'ees_effects': {
            'E': 0.0, 'Ec': 0.15, 'S': 0.05,
        },
        'ees_confidence': {
            'E': 'low', 'Ec': 'low', 'S': 'low',
        },
        'note': (
            'Electrolysis-style interruptible industrial load. '
            'The `flexibility: 0.4` attribute signals to the engine that up to 40% '
            'of this load may be shed during heat-wave or grid-stress disturbances. '
            'Applicable to any study county — load-following flexibility is geography-agnostic. '
            'Capex proxy from IRENA green hydrogen cost data; actual capex varies by technology.'
        ),
    },
}

# Merge new actions into v3 dict
for aid, a in ENERGY_DEMAND_ACTIONS.items():
    acts_v3[aid] = a

print(f"ENERGY_DEMAND actions added: {list(ENERGY_DEMAND_ACTIONS.keys())}")
print(f"Total actions in v3: {len(acts_v3)}  (expected {len(acts_v2)} + 3 = {len(acts_v2)+3})")
assert len(acts_v3) == len(acts_v2) + 3, "Action count mismatch"
print()
for aid in ENERGY_DEMAND_ACTIONS:
    a = acts_v3[aid]
    print(f"  {aid}: unit_scale={a['unit_scale']}, time_to_deploy={a['time_to_deploy']}, "
          f"applicable_counties={len(a['applicable_counties'])}")


ENERGY_DEMAND actions added: ['data_center_hyperscale', 'data_center_campus_phase', 'industrial_load_flexible']
Total actions in v3: 49  (expected 46 + 3 = 49)

  data_center_hyperscale: unit_scale=500, time_to_deploy=2, applicable_counties=115
  data_center_campus_phase: unit_scale=1000, time_to_deploy=3, applicable_counties=115
  industrial_load_flexible: unit_scale=100, time_to_deploy=1, applicable_counties=157


In [6]:

# ── Update material_coefficient_sources.csv ───────────────────────────────────
# Add confidence column to existing rows (mark as 'medium' for documented sources,
# 'low' for flagged ones already in the file), then append ENERGY_DEMAND rows.

coeff_src_v3 = coeff_src.copy()
if 'confidence' not in coeff_src_v3.columns:
    # Assign confidence based on existing notes/sources
    coeff_src_v3['confidence'] = coeff_src_v3.apply(
        lambda r: 'low' if any(x in str(r.get('notes','')) for x in
                               ['estimate','proxy','assume','TODO','judgment','low'])
                  else 'medium',
        axis=1
    )

# New rows for ENERGY_DEMAND actions
new_rows = [
    # data_center_hyperscale
    dict(material='grid_load_mw', value=1.25, unit='MW grid per MW IT load',
         action_type='data_center_hyperscale',
         source='LBNL 2024 US Data Center Energy Usage Report (Shehabi et al.)',
         year=2024, notes='PUE midpoint 1.25; range 1.2-1.3 for hyperscale class', confidence='medium'),
    dict(material='water_consumption_m3_per_mwh', value=1.35, unit='m3/MWh IT energy',
         action_type='data_center_hyperscale',
         source='Siddik et al. (2021) Resources Conservation Recycling 165:105200',
         year=2021, notes='Air-cooled hyperscale midpoint; range 0.9-1.8; liquid-cooled lower',
         confidence='low'),
    dict(material='construction_jobs_per_mw', value=4.0, unit='jobs per MW IT',
         action_type='data_center_hyperscale',
         source='Uptime Institute (2023) Global Data Center Survey — engineering estimate',
         year=2023, notes='Industry range 3-5; no peer-reviewed source available', confidence='low'),
    dict(material='operations_jobs_per_mw', value=0.7, unit='FTE per MW IT',
         action_type='data_center_hyperscale',
         source='Uptime Institute (2023) — engineering estimate',
         year=2023, notes='Industry range 0.5-1.0 for hyperscale', confidence='low'),
    dict(material='county_tax_revenue_usd_per_mw_yr', value=100000,
         unit='USD/MW IT/yr', action_type='data_center_hyperscale',
         source='IDA economic impact disclosures; public incentive records',
         year=2023, notes='Midpoint $50k-250k/MW/yr; highly site-specific', confidence='low'),
    dict(material='land_acres_per_mw', value=1.0, unit='acres per MW IT',
         action_type='data_center_hyperscale',
         source='LBNL 2024 US Data Center Energy Usage Report',
         year=2024, notes='Modern hyperscale range 0.5-1.5 acres/MW', confidence='low'),
    # data_center_campus_phase — same coefficient family
    dict(material='grid_load_mw', value=1.25, unit='MW grid per MW IT load',
         action_type='data_center_campus_phase',
         source='LBNL 2024 US Data Center Energy Usage Report',
         year=2024, notes='Same PUE assumption as data_center_hyperscale', confidence='medium'),
    dict(material='water_consumption_m3_per_mwh', value=1.35, unit='m3/MWh IT',
         action_type='data_center_campus_phase',
         source='Siddik et al. (2021) Resources Conservation Recycling 165:105200',
         year=2021, notes='Same assumption as data_center_hyperscale', confidence='low'),
    dict(material='construction_jobs_per_mw', value=4.0, unit='jobs per MW IT',
         action_type='data_center_campus_phase',
         source='Uptime Institute (2023) — engineering estimate',
         year=2023, notes='Same as hyperscale; phased builds may reduce per-MW cost', confidence='low'),
    dict(material='operations_jobs_per_mw', value=0.7, unit='FTE per MW IT',
         action_type='data_center_campus_phase',
         source='Uptime Institute (2023) — engineering estimate',
         year=2023, notes='Same as hyperscale', confidence='low'),
    dict(material='county_tax_revenue_usd_per_mw_yr', value=100000,
         unit='USD/MW IT/yr', action_type='data_center_campus_phase',
         source='IDA economic impact disclosures', year=2023,
         notes='Same proxy as hyperscale', confidence='low'),
    dict(material='land_acres_per_mw', value=1.0, unit='acres per MW IT',
         action_type='data_center_campus_phase',
         source='LBNL 2024 US Data Center Energy Usage Report',
         year=2024, notes='Same as hyperscale', confidence='low'),
    # industrial_load_flexible
    dict(material='construction_jobs_per_mw', value=2.0, unit='jobs per MW',
         action_type='industrial_load_flexible',
         source='Engineering estimate — no peer-reviewed source',
         year=2024, notes='Electrolysis-style equipment; wide variation by technology',
         confidence='low'),
    dict(material='operations_jobs_per_mw', value=0.3, unit='FTE per MW',
         action_type='industrial_load_flexible',
         source='Engineering estimate — no peer-reviewed source',
         year=2024, notes='Automated electrolysis operations', confidence='low'),
    dict(material='capex_usd_per_mw', value=800000, unit='USD per MW',
         action_type='industrial_load_flexible',
         source='IRENA (2023) Green Hydrogen Cost Reduction Roadmap — proxy',
         year=2023, notes='Electrolysis-class capex proxy; highly technology-dependent',
         confidence='low'),
]

coeff_src_v3 = pd.concat(
    [coeff_src_v3, pd.DataFrame(new_rows)], ignore_index=True
)

coeff_src_v3.to_csv(PROC / 'material_coefficient_sources.csv', index=False)
print(f"material_coefficient_sources.csv: {len(coeff_src)} → {len(coeff_src_v3)} rows")
print(f"  New ENERGY_DEMAND rows added: {len(new_rows)}")
print(f"  Confidence distribution: {coeff_src_v3['confidence'].value_counts().to_dict()}")
n_low = (coeff_src_v3['confidence'] == 'low').sum()
print(f"  Rows flagged confidence='low': {n_low}")


material_coefficient_sources.csv: 29 → 44 rows
  New ENERGY_DEMAND rows added: 15
  Confidence distribution: {'medium': 31, 'low': 13}
  Rows flagged confidence='low': 13


---
## 3 · Coupling Rules

Following the `synergy_groups` pattern in the v2 library, add a top-level
`coupling_rules` block with three rules. All effect magnitudes marked
`confidence: "low"` where judgment-based.


In [7]:

# ── DOE HALEU supply chain data ───────────────────────────────────────────────
# Source: DOE HALEU Availability Program (Infrastructure Investment and Jobs Act 2021,
#         Section 2001; $700M authorized)
# Centrus Energy Corp Piketon OH facility: first HALEU production Dec 2023
# Initial capacity: ~900 kg HALEU per year
# Natrium reactor (345 MW) initial core load: ~5 metric tons HALEU
# Annual reload: ~1–2 metric tons per year
# Source: DOE NE press releases 2023; Centrus Energy SEC filings 2023
# Confidence: low — operational scale not yet demonstrated at full production rate

HALEU_KG_PER_YEAR_INITIAL = 900    # kg/yr from Centrus Piketon facility
HALEU_CONFIDENCE = 'low'           # production rate not yet demonstrated at scale
HALEU_SOURCE = (
    'DOE Office of Nuclear Energy, HALEU Availability Program, '
    'Centrus Energy Corp Piketon OH cascade; '
    'initial production Dec 2023 ~900 kg/yr. '
    'DOE press release 2023-12-13. Confidence low: '
    'full production rate not yet demonstrated.'
)

# BWXT Gillette facility: announced 2025, not yet operational
# oilcity.news 2025-09-25: "BWXT to court Gillette for $500M nuclear fuel manufacturing plant"
# Treat as pre-operational: 0 units/year until operational_year (conservative = 2029)
FUEL_FAB_UNITS_INITIAL  = 0
FUEL_FAB_OP_YEAR        = 2029    # conservative pre-operational baseline
FUEL_FAB_SOURCE = (
    'BWXT Gillette WY $500M nuclear fuel manufacturing facility announced 2025; '
    'oilcity.news 2025-09-25. '
    'Pre-operational baseline = 0 until operational_year 2029 (conservative). '
    'Confidence low: facility not yet permitted or under construction.'
)

print(f"HALEU pool: {HALEU_KG_PER_YEAR_INITIAL} kg/yr  (confidence: {HALEU_CONFIDENCE})")
print(f"HALEU source: {HALEU_SOURCE[:120]}...")
print(f"Fuel fab units/yr: {FUEL_FAB_UNITS_INITIAL} (operational_year: {FUEL_FAB_OP_YEAR})")


HALEU pool: 900 kg/yr  (confidence: low)
HALEU source: DOE Office of Nuclear Energy, HALEU Availability Program, Centrus Energy Corp Piketon OH cascade; initial production Dec...
Fuel fab units/yr: 0 (operational_year: 2029)


In [8]:

# ── nuclear_dc_coupling reasoning ─────────────────────────────────────────────
# transmission_requirement_reduction_pct = 20%
#   Rationale: A hyperscale data center co-located with a baseload nuclear plant
#   (same primary_bus or one branch apart) can be served largely via a dedicated
#   point-to-point connection, bypassing regional transmission congestion.
#   Industry precedent: Microsoft–Constellation Three Mile Island PPA (2023) and
#   Amazon–Talen nuclear DC agreement (2023) involve direct interconnection that
#   reduces AC transmission needs by 15–25%. We use 20% as a conservative midpoint.
#   Confidence: low — actual reduction depends on grid topology and interconnect design.
#
# reliability_credit = 0.20   (on a 0.0–1.0 scale)
#   Rationale: Firm baseload nuclear supply improves the DC's power reliability index
#   by reducing exposure to grid-frequency events and fuel-supply variability.
#   DOE grid reliability studies show nuclear capacity credit ~0.9 vs renewables ~0.3;
#   the 0.2 reliability_credit models a partial benefit from co-location vs full
#   dedicated supply. Confidence: low — credit is regime-specific.

COUPLING_RULES = {
    'nuclear_dc_coupling': {
        'description': (
            'Co-location coupling between an ENERGY_DEMAND action and a firm-clean '
            'supply action (smr_advanced, coal_to_smr, geothermal_utility, or '
            'pumped_hydro with magnitude ≥ 500 MWh) on the same primary_bus '
            'or one branch apart in the synthetic network.'
        ),
        'trigger': {
            'demand_actions': ['data_center_hyperscale', 'data_center_campus_phase'],
            'supply_actions': ['smr_advanced', 'coal_to_smr', 'geothermal_utility'],
            'supply_actions_with_threshold': {
                'pumped_hydro': {'min_magnitude_mwh': 500}
            },
            'spatial_condition': 'same_primary_bus OR one_branch_apart',
        },
        'effects': {
            'transmission_requirement_reduction_pct': {
                'value':      20,
                'unit':       'percent reduction in regional transmission requirement',
                'confidence': 'low',
            },
            'reliability_credit': {
                'value':      0.20,
                'unit':       '0.0–1.0 scale; added to effective reliability index',
                'confidence': 'low',
            },
            'time_to_deploy_pair': {
                'formula':    'max(demand.time_to_deploy, supply.time_to_deploy)',
                'note':       'Co-scheduled builds align operational year to the slower asset.',
                'confidence': 'medium',
            },
        },
        'confidence': 'low',
        'reasoning': (
            'A hyperscale data center anchored to a co-located nuclear plant can '
            'negotiate a direct interconnection, bypassing regional transmission '
            'constraints. Industry precedent (Microsoft–Constellation Three Mile Island '
            'PPA 2023; Amazon–Talen Energy nuclear DC agreement 2023) suggests '
            'transmission requirement reductions of 15–25% from direct interconnect; '
            'we use 20% as a conservative midpoint. The reliability credit of 0.20 '
            'reflects partial benefit: nuclear capacity credit is ~0.9 vs renewables '
            '~0.3 in DOE grid studies, but co-location without full dedicated supply '
            'captures only a fraction of that benefit. Both values are '
            'judgment-based engineering estimates — confidence is explicitly low. '
            'The co-scheduling rule (time_to_deploy = max of the pair) reflects '
            'the practical reality that a DC built before its anchor supply is '
            'operational will draw from the grid anyway, so the coupling benefit '
            'only materializes when both assets are online simultaneously.'
        ),
        'references': [
            'Microsoft–Constellation Three Mile Island Power Purchase Agreement, Sep 2023',
            'Amazon Web Services–Talen Energy Nuclear Data Center Agreement, Mar 2023',
            'DOE Office of Electricity (2023) Nuclear Power Plant Capacity Credit Study',
        ],
    },

    'firm_supply_gap': {
        'description': (
            'Any ENERGY_DEMAND placement where the resolved bus has insufficient '
            'firm capacity: (county_firm_capacity_mw - existing_load_mw) < action_magnitude_mw. '
            'The engine computes and stores deficit_mw on the placed action. '
            'The action is NOT blocked — the tool shows consequences, it does not '
            'forbid choices. The UI must surface deficit_mw prominently.'
        ),
        'trigger': {
            'condition': '(county_firm_capacity_mw - existing_load_mw) < action_magnitude_mw',
            'applies_to_buckets': ['energy_demand'],
        },
        'effect': {
            'attribute': 'deficit_mw',
            'formula':   'action_magnitude_mw - (county_firm_capacity_mw - existing_load_mw)',
            'storage':   'stored on placed action object in state',
        },
        'ui_requirement': (
            'UI must display deficit_mw chip prominently on the county card '
            'and placement ghost. Clicking deficit chip suggests resolving '
            'action categories (energy_generation, energy_storage, nuclear_fuel_cycle).'
        ),
        'confidence': 'high',   # logic is deterministic given inputs
        'note': (
            'Firm capacity is defined as dispatchable generation capacity (nuclear, '
            'gas, hydro, storage) at the county primary_bus from the baseline card. '
            'Renewable capacity (wind, solar) is not counted as firm unless '
            'paired with storage meeting a 4-hour adequacy threshold.'
        ),
    },

    'supply_chain_throughput': {
        'description': (
            'SMR-family actions (smr_advanced, coal_to_smr, haleu_production, '
            'fuel_fabrication) draw from shared annual HALEU and fuel-fabrication '
            'capacity pools. Builds exceeding pool capacity are queued (FIFO by '
            'decision_year), not rejected.'
        ),
        'pools': {
            'HALEU_kg_per_year': {
                'initial_value':   HALEU_KG_PER_YEAR_INITIAL,
                'unit':            'kg HALEU per year',
                'source':          HALEU_SOURCE,
                'year':            2023,
                'confidence':      HALEU_CONFIDENCE,
                'note': (
                    'Centrus Piketon OH cascade producing ~900 kg/yr as of Dec 2023. '
                    'Scale-up to ~6,000 kg/yr planned pending DOE contract extension. '
                    'Use 900 kg/yr as conservative pre-scale-up baseline.'
                ),
            },
            'fuel_fabrication_units_per_year': {
                'initial_value':   FUEL_FAB_UNITS_INITIAL,
                'operational_year': FUEL_FAB_OP_YEAR,
                'unit':            'fuel assemblies per year (facility count proxy)',
                'source':          FUEL_FAB_SOURCE,
                'year':            2025,
                'confidence':      'low',
                'note': (
                    'BWXT Gillette WY facility announced 2025 but not yet operational. '
                    'Conservative baseline = 0 until 2029. '
                    'Increment to 1 facility-equivalent when operational_year is reached.'
                ),
            },
        },
        'smr_haleu_per_build': {
            'value':      5000,
            'unit':       'kg HALEU per Natrium-class (345 MW) initial core',
            'source':     'TerraPower Natrium design specifications (public); '
                          'DOE Advanced Reactor Demonstration Program documentation',
            'year':       2023,
            'confidence': 'low',
            'note': 'Annual reload ~1,000–2,000 kg/yr; initial core load ~5,000 kg',
        },
        'queue_behavior': 'FIFO by decision_year — builds exceeding pool are queued to next available year, not rejected',
        'smr_family_actions': ['smr_advanced', 'coal_to_smr', 'haleu_production', 'fuel_fabrication'],
        'confidence': 'low',
    },
}

print("Coupling rules defined:")
for rule_id, rule in COUPLING_RULES.items():
    print(f"  {rule_id}: confidence={rule.get('confidence','?')}")
    if 'effects' in rule:
        for eff, edata in rule['effects'].items():
            conf = edata.get('confidence','?') if isinstance(edata,dict) else '?'
            print(f"    effect {eff}: confidence={conf}")


Coupling rules defined:
  nuclear_dc_coupling: confidence=low
    effect transmission_requirement_reduction_pct: confidence=low
    effect reliability_credit: confidence=low
    effect time_to_deploy_pair: confidence=medium
  firm_supply_gap: confidence=high
  supply_chain_throughput: confidence=low


---
## 4 · Schema Version and Changelog


In [9]:

SCHEMA_VERSION = "3.0"
CHANGELOG = [
    {
        "version": "3.0",
        "date": "2026-06-10",
        "changes": [
            "All 46 existing actions: placement_scale overridden to 'county' (user-facing siting unit)",
            "All 46 existing actions: resolves_to field added (bus|tract|watershed|county by bucket)",
            "All 46 existing actions: applicable_counties precomputed from crosswalk (ecoregion_area_share ≥ 0.15)",
            "New bucket 'energy_demand' added with 3 new actions: data_center_hyperscale, data_center_campus_phase, industrial_load_flexible",
            "data_center_hyperscale: unit_scale 500 MW, time_to_deploy 2, bus_load_add, DC-BA applicable counties",
            "data_center_campus_phase: unit_scale 1000 MW, time_to_deploy 3, bus_load_add, same applicable counties",
            "industrial_load_flexible: unit_scale 100 MW, time_to_deploy 1, flexibility=0.4, all study counties",
            "coupling_rules block added: nuclear_dc_coupling, firm_supply_gap, supply_chain_throughput",
            "supply_chain_throughput: HALEU pool 900 kg/yr (Centrus Piketon), BWXT Gillette pre-op baseline 0",
            "material_coefficient_sources.csv: confidence column added; 15 new ENERGY_DEMAND coefficient rows",
            "Schema version bumped from taxonomy_v0.1 to 3.0",
        ]
    }
]

print(f"Schema version: {SCHEMA_VERSION}")
print(f"Changelog entry date: {CHANGELOG[0]['date']}")
print(f"Changes logged: {len(CHANGELOG[0]['changes'])}")


Schema version: 3.0
Changelog entry date: 2026-06-10
Changes logged: 11


In [10]:

# ── Assemble and write mw_action_library_v3.json ──────────────────────────────
lib_v3 = {
    'schema_version': SCHEMA_VERSION,
    'changelog':      CHANGELOG,
    'metadata': {
        **meta_v2,
        'schema_version':  SCHEMA_VERSION,
        'action_count':    len(acts_v3),
        'last_updated':    TODAY,
        'county_pivot':    True,
        'new_buckets':     ['energy_demand'],
    },
    'actions':        acts_v3,
    'coupling_rules': COUPLING_RULES,
    'disturbances':   dists_v2,
    '_coefficient_audit': lib_v2.get('_coefficient_audit', {}),
}

out_path = PROC / 'mw_action_library_v3.json'
with open(out_path, 'w') as f:
    json.dump(lib_v3, f, indent=2, default=str)

size_kb = out_path.stat().st_size / 1024
print(f"Saved mw_action_library_v3.json — {size_kb:.0f} KB")
print(f"Total actions: {len(lib_v3['actions'])}")
print(f"schema_version: {lib_v3['schema_version']}")
print(f"coupling_rules: {list(lib_v3['coupling_rules'].keys())}")


Saved mw_action_library_v3.json — 207 KB
Total actions: 49
schema_version: 3.0
coupling_rules: ['nuclear_dc_coupling', 'firm_supply_gap', 'supply_chain_throughput']


---
## Handoff Validation


In [11]:

print("=" * 70)
print("SESSION 0.2 HANDOFF VALIDATION")
print("=" * 70)

checks = []

# Reload from disk for integrity
with open(PROC / 'mw_action_library_v3.json') as f:
    v3 = json.load(f)
acts_check = v3['actions']
coeff_check = pd.read_csv(PROC / 'material_coefficient_sources.csv')

# 1. Action count
n_v2 = len(acts_v2)
n_v3 = len(acts_check)
expected = n_v2 + 3
if n_v3 == expected:
    checks.append(("✓", f"Action count: v2={n_v2}, v3={n_v3} (v2+3 new ENERGY_DEMAND)"))
else:
    checks.append(("✗", f"Action count: v2={n_v2}, v3={n_v3}, expected {expected}"))

# 2. All actions have required new fields
missing_fields = [aid for aid, a in acts_check.items()
                  if not all(k in a for k in ['placement_scale','resolves_to','applicable_counties'])]
if not missing_fields:
    checks.append(("✓", f"All {n_v3} actions have placement_scale, resolves_to, applicable_counties"))
else:
    checks.append(("✗", f"Missing fields on {len(missing_fields)} actions: {missing_fields[:5]}"))

# 3. Spot-check (re-verify from file)
pr_ok = '08117' not in acts_check.get('prairie_restoration',{}).get('applicable_counties',[])
ph_ok = '56005' not in acts_check.get('pumped_hydro',{}).get('applicable_counties',[])
smr_ok = ('56021' in acts_check.get('smr_advanced',{}).get('applicable_counties',[]) and
          '56023' in acts_check.get('smr_advanced',{}).get('applicable_counties',[]))
wind_ok = all(g in acts_check.get('wind_utility',{}).get('applicable_counties',[])
              for g in ['56005','56007','56009'])
c2s_ok = all(g in acts_check.get('coal_to_smr',{}).get('applicable_counties',[])
             for g in ['56037','56005'])
spot_ok = all([pr_ok, ph_ok, smr_ok, wind_ok, c2s_ok])
if spot_ok:
    checks.append(("✓", "5 spot-check county tables verified (sensible exclusions/inclusions)"))
else:
    detail = [f"prairie_rest excl mountain={pr_ok}", f"pumped_hydro excl plains={ph_ok}",
              f"smr incl WY={smr_ok}", f"wind incl Campbell/Carbon/Converse={wind_ok}",
              f"coal_to_smr incl Sweetwater/Campbell={c2s_ok}"]
    checks.append(("✗", f"Spot-check failures: {[d for d in detail if 'False' in d]}"))

# Print spot-check table
print()
print(f"  {'Action':<28} {'N counties':>10}  {'Includes key WY counties?'}")
print(f"  {'-'*65}")
for aid, check_geoids, label in [
    ('prairie_restoration',  [], 'excl Summit/Eagle/Pitkin CO (eco-21-only)'),
    ('pumped_hydro',         [], 'excl Campbell WY (eco-43-only)'),
    ('smr_advanced',         ['56021','56023'], 'incl Laramie + Lincoln WY'),
    ('wind_utility',         ['56005','56007','56009'], 'incl Campbell/Carbon/Converse WY'),
    ('coal_to_smr',          ['56037','56005'], 'incl Sweetwater + Campbell WY'),
]:
    a = acts_check.get(aid, {})
    counties = a.get('applicable_counties', [])
    n = len(counties)
    ok = all(g in counties for g in check_geoids) if check_geoids else True
    flag = "✓" if ok else "✗"
    print(f"  {aid:<28} {n:>10}  {flag} {label}")

# 4. ENERGY_DEMAND bucket
ed_actions = [aid for aid, a in acts_check.items() if a.get('bucket')=='energy_demand']
if len(ed_actions) == 3:
    checks.append(("✓", f"ENERGY_DEMAND bucket present with 3 actions: {ed_actions}"))
else:
    checks.append(("✗", f"ENERGY_DEMAND has {len(ed_actions)} actions (expected 3): {ed_actions}"))

# 5. coupling_rules
cr = v3.get('coupling_rules', {})
expected_rules = {'nuclear_dc_coupling', 'firm_supply_gap', 'supply_chain_throughput'}
if set(cr.keys()) == expected_rules:
    checks.append(("✓", f"coupling_rules block present with all 3 rules"))
else:
    checks.append(("✗", f"coupling_rules: expected {expected_rules}, got {set(cr.keys())}"))

# 6. material_coefficient_sources.csv row count
n_old = len(coeff_src)
n_new = len(coeff_check)
checks.append(("✓" if n_new > n_old else "✗",
               f"material_coefficient_sources.csv: {n_old} → {n_new} rows (+{n_new-n_old})"))

# 7. confidence column and low-confidence count
n_low = (coeff_check['confidence'] == 'low').sum() if 'confidence' in coeff_check.columns else -1
checks.append(("✓" if n_low >= 0 else "✗",
               f"confidence column present; {n_low} rows flagged confidence='low'"))

# 8. schema_version
sv = v3.get('schema_version', 'MISSING')
checks.append(("✓" if sv == '3.0' else "✗", f"schema_version == '{sv}'"))

# Print results
print()
for flag, msg in checks:
    print(f"  [{flag}] {msg}")

n_pass = sum(1 for f, _ in checks if f == "✓")
n_fail = sum(1 for f, _ in checks if f == "✗")
print()
print(f"RESULT: {n_pass} passed, {n_fail} failed")
if n_fail == 0:
    print("Session 0.2 ready for handoff to Session 0.3 (Engine v2 + Golden Fixtures).")
else:
    print("Fix failures before proceeding.")


SESSION 0.2 HANDOFF VALIDATION

  Action                       N counties  Includes key WY counties?
  -----------------------------------------------------------------
  prairie_restoration                  98  ✓ excl Summit/Eagle/Pitkin CO (eco-21-only)
  pumped_hydro                         78  ✓ excl Campbell WY (eco-43-only)
  smr_advanced                         85  ✓ incl Laramie + Lincoln WY
  wind_utility                        157  ✓ incl Campbell/Carbon/Converse WY
  coal_to_smr                          67  ✓ incl Sweetwater + Campbell WY

  [✓] Action count: v2=46, v3=49 (v2+3 new ENERGY_DEMAND)
  [✓] All 49 actions have placement_scale, resolves_to, applicable_counties
  [✓] 5 spot-check county tables verified (sensible exclusions/inclusions)
  [✓] ENERGY_DEMAND bucket present with 3 actions: ['data_center_hyperscale', 'data_center_campus_phase', 'industrial_load_flexible']
  [✓] coupling_rules block present with all 3 rules
  [✓] material_coefficient_sources.csv: 29 → 44 